In [ ]:


# ============================================================
# ASSIGNMENT 6: TRANSFER LEARNING USING RESNET50
# Dataset: MNIST (10 classes)
# Pre-trained CNN: ResNet50 with ImageNet weights
# ============================================================

%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.datasets import mnist
from IPython.display import display

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)


# ============================================================
# 1. LOAD AND PREPROCESS THE DATASET
# ============================================================

(x_train, y_train), (x_test, y_test) = mnist.load_data()

rng = np.random.default_rng(0)

# Select subsets to reduce training time
tr = rng.choice(len(x_train), 6000, replace=False)
te = rng.choice(len(x_test), 1500, replace=False)


def prep(x):
    # Resize grayscale images to 32x32
    x = tf.image.resize(
        x[..., None].astype("float32"),
        (32, 32)
    ).numpy()

    # Convert grayscale images to RGB
    x = np.repeat(x, 3, axis=-1)

    # Apply ResNet50 preprocessing
    return preprocess_input(x)


X_tr = prep(x_train[tr])
y_tr = y_train[tr]

X_te = prep(x_test[te])
y_te = y_test[te]

print("Training data shape:", X_tr.shape)
print("Testing data shape:", X_te.shape)


# ============================================================
# 2. LOAD PRE-TRAINED RESNET50
# ============================================================

base = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(32, 32, 3)
)

print("Base model layers:", len(base.layers))
print("Base model parameters:", f"{base.count_params():,}")


# ============================================================
# 3. FREEZE THE BASE MODEL
# ============================================================

base.trainable = False


# ============================================================
# 4. ADD A CUSTOM CLASSIFICATION HEAD
# ============================================================

inputs = keras.Input(shape=(32, 32, 3))

x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(10, activation="softmax")(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print(
    "Trainable parameters (classifier head):",
    f"{sum(int(np.prod(w.shape)) for w in model.trainable_weights):,}"
)


# ============================================================
# 5. TRAIN THE CLASSIFICATION HEAD (8 EPOCHS)
# ============================================================

h1 = model.fit(
    X_tr,
    y_tr,
    validation_data=(X_te, y_te),
    epochs=8,
    batch_size=64,
    verbose=2
)

stage1_loss, stage1_acc = model.evaluate(
    X_te, y_te, verbose=0
)

print(f"\nStage-1 test loss: {stage1_loss:.4f}")
print(f"Stage-1 test accuracy: {stage1_acc:.4f}")


# ============================================================
# 6. UNFREEZE THE LAST CONVOLUTIONAL BLOCK
# ============================================================

base.trainable = True

for layer in base.layers:
    layer.trainable = (
        layer.name.startswith("conv5")
        and not isinstance(layer, layers.BatchNormalization)
    )

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print(
    "Trainable parameters after fine-tuning setup:",
    f"{sum(int(np.prod(w.shape)) for w in model.trainable_weights):,}"
)


# ============================================================
# 7. FINE-TUNE THE MODEL (6 EPOCHS)
# ============================================================

h2 = model.fit(
    X_tr,
    y_tr,
    validation_data=(X_te, y_te),
    epochs=6,
    batch_size=64,
    verbose=2
)


# ============================================================
# 8. FINAL MODEL EVALUATION
# ============================================================

final_loss, final_acc = model.evaluate(
    X_te, y_te, verbose=0
)

print(f"\nFinal test loss: {final_loss:.4f}")
print(f"Final test accuracy: {final_acc:.4f}")


# ============================================================
# 9. COMBINE BOTH TRAINING HISTORIES
# ============================================================

loss_all = h1.history["loss"] + h2.history["loss"]
val_loss_all = h1.history["val_loss"] + h2.history["val_loss"]

acc_all = h1.history["accuracy"] + h2.history["accuracy"]
val_acc_all = h1.history["val_accuracy"] + h2.history["val_accuracy"]

n1 = len(h1.history["loss"])
epochs = range(1, len(loss_all) + 1)


# ============================================================
# 10. PLOT LOSS AND ACCURACY GRAPHS
# ============================================================

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

# ---------------- LOSS GRAPH ----------------

ax[0].plot(epochs, loss_all, label="train")
ax[0].plot(epochs, val_loss_all, label="validation")

# Dashed separator between training stages
ax[0].axvline(
    n1 + 0.5,
    color="gray",
    linestyle="--"
)

ymin, ymax = ax[0].get_ylim()

ax[0].text(
    n1 + 0.65,
    ymin + 0.02 * (ymax - ymin),
    "fine-tuning ->",
    fontsize=8
)

ax[0].set_title("Loss (frozen base, then fine-tuning)")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss")
ax[0].set_xticks(range(2, len(loss_all) + 1, 2))
ax[0].legend()
ax[0].grid(alpha=0.3)


# ---------------- ACCURACY GRAPH ----------------

ax[1].plot(epochs, acc_all, label="train")
ax[1].plot(epochs, val_acc_all, label="validation")

ax[1].axvline(
    n1 + 0.5,
    color="gray",
    linestyle="--"
)

ymin, ymax = ax[1].get_ylim()

ax[1].text(
    n1 + 0.65,
    ymin + 0.02 * (ymax - ymin),
    "fine-tuning ->",
    fontsize=8
)

ax[1].set_title("Accuracy (frozen base, then fine-tuning)")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Accuracy")
ax[1].set_xticks(range(2, len(acc_all) + 1, 2))
ax[1].legend()
ax[1].grid(alpha=0.3)


# ============================================================
# 11. SAVE AND DISPLAY THE GRAPH
# ============================================================

fig.tight_layout()

fig.savefig(
    "a6_curves.png",
    dpi=150,
    bbox_inches="tight"
)

# Explicitly display the figure once in Google Colab
display(fig)

plt.close(fig)


# ============================================================
# 12. FINAL OUTPUT
# ============================================================

print("\nAssignment 6 completed successfully!")
print("Stage 1: Frozen base training - 8 epochs")
print("Stage 2: Fine-tuning - 6 epochs")
print("Total epochs:", len(loss_all))
print("Graph saved as: a6_curves.png")



Training data shape: (6000, 32, 32, 3)
Testing data shape: (1500, 32, 32, 3)
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 9s 0us/step
Base model layers: 175
Base model parameters: 23,587,712
Trainable parameters (classifier head): 527,114
Epoch 1/8
94/94 - 87s - 927ms/step - accuracy: 0.7152 - loss: 0.9644 - val_accuracy: 0.8880 - val_loss: 0.3692
Epoch 2/8
94/94 - 57s - 606ms/step - accuracy: 0.8642 - loss: 0.3962 - val_accuracy: 0.8987 - val_loss: 0.3069
Epoch 3/8
94/94 - 81s - 857ms/step - accuracy: 0.8980 - loss: 0.3035 - val_accuracy: 0.9133 - val_loss: 0.2673
Epoch 4/8
94/94 - 53s - 563ms/step - accuracy: 0.9125 - loss: 0.2589 - val_accuracy: 0.9020 - val_loss: 0.3028
Epoch 5/8
94/94 - 56s - 591ms/step - accuracy: 0.9263 - loss: 0.2190 - val_accuracy: 0.9087 - val_loss: 0.2798
Epoch 6/8
94/94 - 56s - 599ms/step - accuracy: 0.9367 - loss: 0.1954 - val_accuracy: 0.9127 - val_loss: 0.2745
Epoch 7/8
94/94 - 52s - 556ms/step - accuracy: 0.9432 - loss: 0.1622 - val_accuracy: 0.9153 - val_los